In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
import json

In [2]:
sample_names = [
    "C4",
    "finepdfs",
    "fineweb",
    "RedPajama",]

In [ ]:
# Creating html file with all texts from all samples and the eval table with some pre-filled metrics
max_text_length = 2000

html_content = "<html><body>"
csv_rows = []

for sample_name in sample_names:
    with open(f"samples\\{sample_name}_dataset.json", 'r') as f:
        data = json.load(f)
        html_content += f"<h2>Samples from {sample_name}</h2>"
        for i, sample in enumerate(data):
            html_content += f"<h3>Sample {i+1}</h3>"
            if sample_name == 'RedPajama':
                meta = sample['meta']
                # meta is "{\"url\": \"<url>\"[...]"
                url = meta.split('{\"url\": \"')[1].split('\"')[0]
                text = sample['raw_content'] 
            else:
                url = sample['url']
                text = sample['text']
            num_chars = len(text)
            # infer rough number of sentences by counting punctuation marks
            num_sentences = text.count('.')+text.count('!')+text.count('?')
            if num_chars > max_text_length:
                text = text[:max_text_length] + "..."
            html_content += f"<p><strong>URL:</strong> <a href='{url}' target='_blank'>{url}</a></p>"
            html_content += f"<p><strong>Text {i+1}:</strong> {text}</p>"

            # Build the row for table with pre-filled metrics and blank slots for amanual evaluations
            row = {
                'dataset_source': sample_name,
                'document_index': sample['document_index'],
                'topic': '',               
                'quality (1-10)': '',      
                'correctly_cleaned': '',   
                'private': 0,
                'harmful': 0,  
                'num_characters': num_chars,
                'num_sentences': num_sentences
            }
            csv_rows.append(row)

with open("samples.html", "w") as f:
    f.write(html_content)

df_eval = pd.DataFrame(csv_rows)
output_filename = "sample_evaluation.csv"
df_eval.to_csv(output_filename, index=False, sep=";",encoding='utf-8')